In [1]:
import pandas as pd
import numpy as np
import os

In [3]:
balls = pd.read_csv("ball_by_ball.csv")
matches = pd.read_csv("matches.csv")

# Merge match information

In [4]:
balls = balls.merge(
    matches[[
        "match_id",
        "first_inn_revised_overs",
        "second_inn_revised_overs",
        "revised_target"
    ]],
    on="match_id",
    how="left"
)

# Calculate innings score before each delivery

In [5]:
balls["runs"] = pd.to_numeric(
    balls["runs"], errors="coerce"
).fillna(0)

In [6]:
balls["is_wicket"] = (
    balls["is_wicket"].astype(str).str.lower() == "true"
)

In [7]:
balls = balls.sort_values(
    ["match_id", "innings", "innings_ball_num"]
)

In [11]:
group = balls.groupby(["match_id", "innings"])
group.head()

,match_id,innings,overid,over_ball_num,innings_ball_num,batting_team_id,bowling_team_id,striker_id,striker_name,non_striker_id,...,stroke,delivery_style,over_or_around,on_or_off,delivery_ist_time,first_inn_revised_overs,second_inn_revised_overs,revised_target,score_before,wickets_before
0,13511,1,1,1,1,22,28,1187,Jason Roy,6273,...,Cover Drive,Inswing,Over,Off-side,2022-01-23 00:00:00 UTC,NaN,NaN,NaN,0,0
1,13511,1,1,2,2,22,28,1187,Jason Roy,6273,...,Flick,Inswing,Over,On-side,2022-01-23 00:00:00 UTC,NaN,NaN,NaN,0,0
2,13511,1,1,3,3,22,28,1187,Jason Roy,6273,...,Square Cut,Inswing,Over,Off-side,2022-01-23 00:00:00 UTC,NaN,NaN,NaN,0,0
3,13511,1,1,4,4,22,28,1187,Jason Roy,6273,...,Backfoot defensive,Inswing,Over,On-side,2022-01-23 00:00:00 UTC,NaN,NaN,NaN,0,0
4,13511,1,1,5,5,22,28,1187,Jason Roy,6273,...,Pull shot,Seamed In,Over,On-side,2022-01-23 00:00:00 UTC,NaN,NaN,NaN,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
200947,55561,2,1,1,1,78,72,17910,Mohammad Akram,6448,...,Square Cut,Out Swing,Over,Off-side,2026-11-01 16:08:59 UTC,NaN,NaN,NaN,0,0
200948,55561,2,1,2,2,78,72,17910,Mohammad Akram,6448,...,Pull shot,Off Cutter,Over,On-side,2026-11-01 16:09:35 UTC,NaN,NaN,NaN,4,0
200949,55561,2,1,3,3,78,72,17910,Mohammad Akram,6448,...,Square Cut,Seamed Out,Over,Off-side,2026-11-01 16:10:19 UTC,NaN,NaN,NaN,8,0
200950,55561,2,1,4,4,78,72,17910,Mohammad Akram,6448,...,Pull shot,Seamed In,Over,On-side,2026-11-01 16:11:01 UTC,NaN,NaN,NaN,12,0


In [9]:
balls["score_before"] = group["runs"].cumsum() - balls["runs"]
balls["wickets_before"] = (
    group["is_wicket"].cumsum().astype(int)
    - balls["is_wicket"].astype(int)
)

# Set innings overs

In [16]:
balls["max_overs"] = np.where(
    balls["innings"] == 1,
    balls["first_inn_revised_overs"],
    balls["second_inn_revised_overs"]
)


In [13]:
balls["max_overs"] = pd.to_numeric(
    balls["max_overs"], errors="coerce"
).fillna(20)

# Balls remaining

In [14]:
balls["balls_remaining"] = (
    balls["max_overs"] * 6 - balls["innings_ball_num"]
).clip(lower=0)


# Calculate first innings totals

In [17]:
first_scores = (
    balls[balls["innings"] == 1]
    .groupby("match_id")["runs"]
    .sum()
    .add(1)
)

In [18]:

balls["target"] = balls["match_id"].map(first_scores)

# Use revised target when available

In [19]:
revised = pd.to_numeric(
    balls["revised_target"], errors="coerce"
)


In [20]:
balls["target"] = revised.where(
    revised.notna() & (revised > 0),
    balls["target"]
)

# Calculate required run rate in second innings

In [21]:
balls["runs_needed"] = (
    balls["target"] - balls["score_before"]
).clip(lower=0)

balls["required_rr"] = np.where(
    (balls["innings"] == 2) &
    (balls["balls_remaining"] > 0),
    balls["runs_needed"] * 6 /
    balls["balls_remaining"],
    0
)


# Define pressure situations

In [22]:
balls["pressure"] = (
    (balls["overid"] >= 16) |
    (balls["wickets_before"] >= 6) |
    (balls["required_rr"] >= 10) |
    (
        (balls["wickets_before"] >= 5) &
        (balls["balls_remaining"] <= 60)
    ) |
    (
        (balls["balls_remaining"] <= 30) &
        (balls["required_rr"] >= 9)
    )
)


# Save results

In [24]:
os.makedirs("processed", exist_ok=True)

balls.to_csv(
    "processed/pressure_analysis.csv",
    index=False
)

print("Pressure analysis completed!")
print("\nPressure deliveries:")
print(balls["pressure"].value_counts())

print("\nSaved:pressure_analysis.csv")

Pressure analysis completed!

Pressure deliveries:
pressure
False    119563
True      81436
Name: count, dtype: int64

Saved:pressure_analysis.csv
